# Aurink 非 LLM 小模型實測 (2026-11-25)
全面掃描 Hugging Face 後,對通過篩選的模型逐一實測。需求:去口水詞 / 標點還原 / 錯字修正 / 輕量潤稿,zh-TW 優先,可上裝置,商用授權。

**只跑推理,全部 CPU 即可。**


In [ ]:
# Cell 0: 安裝依賴
!pip -q install -U transformers sentencepiece huggingface_hub onnxruntime
!pip -q install punctuators 2>/dev/null || echo "punctuators install failed (1-800-BAD-CODE 會改用備援路徑)"
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "(本測試用 CPU 即可)")


## 測試輸入(Aurink 代表性案例 + 對照組)


In [ ]:
# Cell 1: 共用測試輸入
INPUTS = {
  "zh_tw_unpunct": "我們明天早上九點開會記得帶報告還有上個月的數據",          # 無標點 zh-TW
  "zh_tw_filler":  "那個我們明天的會議呃改到下午三點然後記得帶那個報告",        # 口水詞 zh-TW
  "zh_trad_typo":  "為了降低少子化，政府可以堆動獎勵生育的政策。",              # 繁體錯字(堆動->推動)
  "zh_trad_typo2": "我們禮拜三在討論這個問題",                                  # 繁體錯字(在->再)
  "zh_simp":       "我已经把文件发给你了请注意查收",                            # 簡體輸入(觀察是否被改壞)
  "zh_name_ctl":   "我朋友叫做楊知頤他在學校教書",                              # 對照:人名不可動
  "en_filler":     "so um I think we should like maybe move the meeting to to three pm",  # 英文口水詞
  "en_grammar":    "their going to review the proposal on Thursday",          # 文法(they're)
  "en_grammar2":   "I has finished the report yesterday and she don't mind",  # 文法
  "en_unpunct":    "can you send me the report before friday thanks",         # 無標點 en
  "en_name_ctl":   "my colleague Jon will send the invoice for Aurink",       # 對照:人名/產品名不可動
  "ja_unpunct":    "明日の会議は十時からです資料を持ってきてください",            # 無標點 ja
  "es_unpunct":    "mañana tenemos una reunión a las nueve no olvides traer el informe",  # 無標點 es
}
import time
def show(title, pairs):
    print("\n" + "="*70 + "\n" + title + "\n" + "="*70)
    for k, out in pairs:
        print(f"[{k}]\n  IN : {INPUTS[k]}\n  OUT: {out}")


## 1. Horizon-Labs/punctuation-restoration-small(140M, Apache-2.0, 88 語言含中/日/西)


In [ ]:
# Cell 2
try:
    from huggingface_hub import hf_hub_download
    import importlib.util
    spec = importlib.util.spec_from_file_location("punctuate", hf_hub_download("Horizon-Labs/punctuation-restoration-small", "punctuate.py"))
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
    p_small = mod.Punctuator("Horizon-Labs/punctuation-restoration-small")
    keys = ["zh_tw_unpunct","zh_tw_filler","zh_name_ctl","en_unpunct","en_name_ctl","ja_unpunct","es_unpunct"]
    t0=time.time(); outs=[(k, p_small(INPUTS[k])) for k in keys]; dt=time.time()-t0
    show(f"Horizon punctuation-restoration-small ({dt:.1f}s for {len(keys)} 句)", outs)
except Exception as e:
    import traceback; traceback.print_exc()


## 2. Horizon-Labs/multilingual-punctuation-truecase-base(307M, Apache-2.0, 88 語言,加大小寫還原)


In [ ]:
# Cell 3
try:
    p_tc = mod.Punctuator("Horizon-Labs/multilingual-punctuation-truecase-base")
    keys = ["zh_tw_unpunct","en_unpunct","en_name_ctl","ja_unpunct","es_unpunct"]
    t0=time.time(); outs=[(k, p_tc(INPUTS[k])) for k in keys]; dt=time.time()-t0
    show(f"Horizon multilingual-punctuation-truecase-base ({dt:.1f}s)", outs)
except Exception as e:
    import traceback; traceback.print_exc()


## 3. 1-800-BAD-CODE/xlm-roberta_punctuation_fullstop_truecase(Apache-2.0, ONNX, 47 語言)


In [ ]:
# Cell 4
try:
    from punctuators.models import PunctCapSegModelONNX
    m1800 = PunctCapSegModelONNX.from_pretrained("1-800-BAD-CODE/xlm-roberta_punctuation_fullstop_truecase")
    keys = ["zh_tw_unpunct","en_unpunct","en_name_ctl","ja_unpunct","es_unpunct"]
    t0=time.time()
    outs=[]
    for k in keys:
        r = m1800.infer([INPUTS[k]])
        outs.append((k, r[0][0] if r and r[0] else str(r)))
    show(f"1-800-BAD-CODE xlm-roberta punct ({time.time()-t0:.1f}s)", outs)
except Exception as e:
    import traceback; traceback.print_exc()


## 4. stillerman/fdt-disfluency-distilbert-66m-v5(英文去口水詞,KEEP/DELETE 標註器;Apache-2.0 但訓練資料含 NC,部署前需處理)


In [ ]:
# Cell 5
try:
    from transformers import AutoTokenizer, AutoModelForTokenClassification
    import torch
    M="stillerman/fdt-disfluency-distilbert-66m-v5"
    tok=AutoTokenizer.from_pretrained(M); mdl=AutoModelForTokenClassification.from_pretrained(M); mdl.eval()
    id2lab=mdl.config.id2label
    def fdt_clean(text):
        words=text.split()
        enc=tok(words, is_split_into_words=True, return_tensors="pt", truncation=True)
        with torch.no_grad(): logits=mdl(**enc).logits[0]
        pred=logits.argmax(-1).tolist(); wids=enc.word_ids()
        labs={}
        for p,w in zip(pred,wids):
            if w is None: continue
            labs.setdefault(w,[]).append(id2lab[p])
        out=[]
        for i,wd in enumerate(words):
            ls=labs.get(i,["KEEP"]); lab=max(set(ls), key=ls.count)
            if lab=="DELETE": continue
            if lab=="KEEP_STRIP_COMMA": wd=wd.rstrip(",")
            if lab=="KEEP_CAPITALIZE": wd=wd.capitalize()
            out.append(wd)
        return " ".join(out)
    keys=["en_filler","en_name_ctl","en_unpunct"]
    t0=time.time(); outs=[(k, fdt_clean(INPUTS[k])) for k in keys]; dt=time.time()-t0
    show(f"stillerman fdt-disfluency-66m ({dt:.1f}s)", outs)
except Exception as e:
    import traceback; traceback.print_exc()


## 5. shibing624/macbert4csc-base-chinese(中文錯字,102M, Apache-2.0;簡體資料訓練,觀察繁體是否被改壞)


In [ ]:
# Cell 6
try:
    from transformers import AutoTokenizer, BertForMaskedLM
    M="shibing624/macbert4csc-base-chinese"
    tok=AutoTokenizer.from_pretrained(M); mdl=BertForMaskedLM.from_pretrained(M); mdl.eval()
    def csc(text):
        enc=tok(text, return_tensors="pt")
        with torch.no_grad(): logits=mdl(**enc).logits[0]
        ids=enc["input_ids"][0]
        out=[]
        for i,(lg,oid) in enumerate(zip(logits, ids)):
            if i==0 or i==len(ids)-1: continue
            pid=lg.argmax().item()
            out.append(tok.decode([pid if pid!=tok.unk_token_id else oid]))
        return "".join(out)
    keys=["zh_trad_typo","zh_trad_typo2","zh_simp","zh_name_ctl","zh_tw_filler"]
    t0=time.time(); outs=[(k, csc(INPUTS[k])) for k in keys]; dt=time.time()-t0
    show(f"macbert4csc ({dt:.1f}s)", outs)
except Exception as e:
    import traceback; traceback.print_exc()


## 6. CodeTed/Chinese_Spelling_Correction_T5(中文錯字 T5, Apache-2.0,卡片範例為繁體)


In [ ]:
# Cell 7
try:
    from transformers import AutoTokenizer, T5ForConditionalGeneration
    M="CodeTed/Chinese_Spelling_Correction_T5"
    tok=AutoTokenizer.from_pretrained(M); mdl=T5ForConditionalGeneration.from_pretrained(M); mdl.eval()
    def ct5(text):
        ids=tok("糾正句子裡的錯字: "+text, return_tensors="pt").input_ids
        with torch.no_grad(): out=mdl.generate(ids, max_length=256)
        return tok.decode(out[0], skip_special_tokens=True)
    keys=["zh_trad_typo","zh_trad_typo2","zh_simp","zh_name_ctl"]
    t0=time.time(); outs=[(k, ct5(INPUTS[k])) for k in keys]; dt=time.time()-t0
    show(f"CodeTed CSC T5 ({dt:.1f}s)", outs)
except Exception as e:
    import traceback; traceback.print_exc()


## 7. shibing624/mengzi-t5-base-chinese-correction(中文語法糾錯 T5, 228M, Apache-2.0)


In [ ]:
# Cell 8
try:
    M="shibing624/mengzi-t5-base-chinese-correction"
    tok=AutoTokenizer.from_pretrained(M); mdl=T5ForConditionalGeneration.from_pretrained(M); mdl.eval()
    def mz5(text):
        ids=tok(text, return_tensors="pt").input_ids
        with torch.no_grad(): out=mdl.generate(ids, max_length=256)
        return tok.decode(out[0], skip_special_tokens=True)
    keys=["zh_trad_typo","zh_trad_typo2","zh_tw_filler","zh_name_ctl"]
    t0=time.time(); outs=[(k, mz5(INPUTS[k])) for k in keys]; dt=time.time()-t0
    show(f"mengzi-t5 correction ({dt:.1f}s)", outs)
except Exception as e:
    import traceback; traceback.print_exc()


## 8. visheratin/t5-efficient-mini-grammar-correction(英文文法,31M, MIT)


In [ ]:
# Cell 9
try:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    M="visheratin/t5-efficient-mini-grammar-correction"
    tok=AutoTokenizer.from_pretrained(M); mdl=AutoModelForSeq2SeqLM.from_pretrained(M); mdl.eval()
    def gec(text):
        ids=tok(text, return_tensors="pt").input_ids
        with torch.no_grad(): out=mdl.generate(ids, max_length=256)
        return tok.decode(out[0], skip_special_tokens=True)
    keys=["en_grammar","en_grammar2","en_filler","en_name_ctl"]
    t0=time.time(); outs=[(k, gec(INPUTS[k])) for k in keys]; dt=time.time()-t0
    show(f"visheratin t5-mini GEC ({dt:.1f}s)", outs)
except Exception as e:
    import traceback; traceback.print_exc()


## 9. oliverguhr/spelling-correction-english-base(英文拼字,MIT,實驗性)


In [ ]:
# Cell 10
try:
    from transformers import pipeline
    fix=pipeline("text2text-generation", model="oliverguhr/spelling-correction-english-base")
    keys=["en_grammar","en_unpunct","en_name_ctl"]
    t0=time.time(); outs=[(k, fix(INPUTS[k], max_length=256)[0]["generated_text"]) for k in keys]; dt=time.time()-t0
    show(f"oliverguhr spelling en ({dt:.1f}s)", outs)
except Exception as e:
    import traceback; traceback.print_exc()


## 10. ElToluene/japanese-punctuation-restoration(日文專用標點,Apache-2.0)


In [ ]:
# Cell 11
try:
    from transformers import pipeline
    jp=pipeline("token-classification", model="ElToluene/japanese-punctuation-restoration")
    print(jp(INPUTS["ja_unpunct"]))
except Exception as e:
    import traceback; traceback.print_exc()
